In [1]:
!conda env list



# conda environments:
#
# * -> active
# + -> frozen
base                 *   /home/sayan/miniconda3
pytorch-env              /home/sayan/miniconda3/envs/pytorch-env



In [2]:
import os
print(os.environ['CONDA_DEFAULT_ENV'])

base


In [3]:
import datetime 
from functools import partial 
from glob import glob
import json 
import math 
import multiprocessing 
import os
from pathlib import Path
import random
from typing import Any, Dict, Optional 

In [4]:
import matplotlib.pyplot as plt
import numpy as np
from distinctipy import distinctipy
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

In [5]:
import pandas as pd
pd.set_option('max_colwidth', None)  # Do not truncate the contents of cells in the DataFrame
pd.set_option('display.max_rows', None)  # Display all rows in the DataFrame
pd.set_option('display.max_columns', None)  # Display all columns in the DataFrame


In [6]:
import torch
from torch.amp import autocast
from torch.cuda.amp import GradScaler
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchtnt.utils import get_module_summary
import torchvision
torchvision.disable_beta_transforms_warning()
from torchvision.tv_tensors import BoundingBoxes, Mask
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks
import torchvision.transforms.v2  as transforms
from torchvision.transforms.v2 import functional as TF

# Import Mask R-CNN
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


In [7]:
seed = 1234 
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
set_seed(seed) 

In [8]:
def get_torch_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

device = get_torch_device()
dtype = torch.float32
device, dtype

(device(type='cuda'), torch.float32)

In [9]:
# The name for the project
project_name = f"pytorch-mask-r-cnn-instance-segmentation"

# The path for the project folder
project_dir = Path(f"./{project_name}/")

# Create the project directory if it does not already exist
project_dir.mkdir(parents=True, exist_ok=True)

# Define path to store datasets
dataset_dir = Path("./Datasets/")
# Create the dataset directory if it does not exist
dataset_dir.mkdir(parents=True, exist_ok=True)

pd.Series({
    "Project Directory:": project_dir, 
    "Dataset Directory:": dataset_dir
}).to_frame().style.hide(axis='columns')

Project Directory:,pytorch-mask-r-cnn-instance-segmentation
Dataset Directory:,Datasets


In [10]:
# Set the name of the dataset
dataset_name = 'pytorch-for-information-extraction'

# Construct the GitHub repository name 
gh_repo = f'cj-mills/{dataset_name}'

# Create the path to the directory where the dataset will be extracted
dataset_path = Path(f'{dataset_dir}/{dataset_name}/code/datasets/detection/student-id/')

pd.Series({
    "GitHub Repository:": gh_repo, 
    "Dataset Path:": dataset_path
}).to_frame().style.hide(axis='columns')

GitHub Repository:,cj-mills/pytorch-for-information-extraction
Dataset Path:,Datasets/pytorch-for-information-extraction/code/datasets/detection/student-id


In [ ]:
#Dataset
!git clone {f 'https://github.com/cj-mills/pytorch-for-information-extraction'} {Datasets/pytorch-for-information-extraction/code/datasets/detection/student-id}

In [ ]:
# import os
# print(os.getcwd())

In [15]:
def get_img_files(dataset_path):
    image_data = []

    for filename in os.listdir(dataset_path):
        if filename.endswith(".jpg") or filename.endswith(".png"):
            img_path = os.path.join(dataset_path, filename)
            img = Image.open(img.path).convert("RGB")
            img_array = np.array(img)
            image_data.append(img_array)
        image_data = np.array(image_data)
    return image_data

In [16]:
img_file_paths = get_img_files(dataset_path)

# Get a list of JSON files in the dataset
annotation_file_paths = list(dataset_path.glob('*.json'))

# Display the names of the folders using a Pandas DataFrame
pd.DataFrame({"Image File": [file.name for file in img_file_paths], 
              "Annotation File":[file.name for file in annotation_file_paths]}).head()

FileNotFoundError: [Errno 2] No such file or directory: 'Datasets/pytorch-for-information-extraction/code/datasets/detection/student-id'

In [13]:
img_dict = {file.stem : file for file in img_file_paths}
print(f"Number of Images: {len(img_dict)}")

pd.Dataframe.from_dict(img_dict, orient = 'index').head()

NameError: name 'img_file_paths' is not defined

In [ ]:
cls_dataframes = (pd.read_json(f, orient = 'index').transpose() for i in tqdm(annotation_file_paths))
annotation_df = pd.concat(cls_dataframes, ignore_index = False)

annotation_df['index'] = annotation_df.apply(lambda row: row[imagePath'].split('.')[0], axis = 1)
annotation_df = annotation_df.set_index('index')

# Keep only the rows that correspond to the filenames in the 'img_dict' dictionary
annotation_df = annotation_df.loc[list(img_dict.keys())]

# Print the first 5 rows of the DataFrame
annotation_df.head()


In [ ]:
shapes_df = annotation_df['shapes'].explode().to_frame().shapes.apply(pd.Series)
class_names = shapes_df['label'].unique().tolist()

pd.DataFrame(class_names)

In [ ]:
class_counts = shapes_df['label'].value_counts()

# Plot the distribution
class_counts.plot(kind='bar')
plt.title('Class distribution')
plt.ylabel('Count')
plt.xlabel('Classes')
plt.xticks(range(len(class_counts.index)), class_names, rotation=75)  # Set the x-axis tick labels
plt.show()